### **While the running is very slow, other users might also use the GPU**

You can open another `.ipynb` and run things in parallel while your Painn model training is running, but there are a couple of important points to keep in mind since you’re on the remote workstation (boltzmann):


Things to check before running another notebook:

	1.	GPU usage
	- Your current run is already using `gpu_num=0`.
	- If you open another notebook and also set it to use GPU 0, **they will fight for the same GPU memory → one of them might crash or slow down**.
	- To avoid that, assign the second job to a different GPU **(gpu_num=1, gpu_num=2, etc.)** if available.
	- Run `nvidia-smi` in the terminal to see how many GPUs are free.

	2.	CPU & Memory usage
	- If your workstation has multiple CPUs and enough RAM, you can run another job in parallel (even on CPU).
	- But training is heavy, so if you run too many jobs, they’ll slow each other down.

	3.	Separate Conda Environments
	- If the second notebook uses the same Conda env, that’s fine.
	- Just make sure the kernel you select in Jupyter matches the right environment.

```
run_wandb_experiment(
    struct_type="unrelaxed",
    model_type="CGCNN",
    gpu_num=1,   # use GPU 1 instead of 0
    obs_budget=1,
    training_fraction=0.125,
    training_seed=1
)
```

If you only have 1 GPU, you can still run lightweight tasks in another notebook (like data analysis, plotting results, testing preprocessing code), but avoid starting another training job.


### On a shared workstation:

1. Check GPU availability first

Run in terminal (inside VS Code or SSH):

```
nvidia-smi
```


This shows:

	- Which GPUs are available (GPU 0, GPU 1, …).
	- Who is using them (look under the processes section → you’ll see usernames + how much GPU memory they’re using).
	- How much GPU memory is free.


2. Coordinate with labmates
	- If your colleague is already using GPU 1, and you’re on GPU 0, that’s fine.
	- But don’t grab the same GPU unless you both agree — it can crash both of your jobs.


3. Best practices in shared GPU workstation

	- Always check nvidia-smi before starting a run.
	- If you have access, you can also run:

```
watch -n 1 nvidia-smi
```


This updates every second, so you can see in real time when a GPU frees up.

	- If multiple GPUs are busy, you can either:
	- Wait until one is free, or
	- Run your experiment on CPU only (much slower, but safe).

**Check `P2.png`**


Here’s what it means
	- GPU 0 (RTX 4090) is the only GPU in the machine.
	- It has 24 GB total memory.
Currently:
	- One python process is using ~20.6 GB → probably your labmate’s job.
	- One process from your env (...Perovskite_ML_Environment/bin/python) is using ~632 MB → that’s your job.
	- **GPU utilization is 100%, meaning the GPU is fully loaded.**

Because there’s only one GPU, and it’s already maxed out, starting another heavy training job will slow both of you down or even cause out-of-memory errors.

What you can do:

	1.	Coordinate with your labmate
	- Ask if you can run another job simultaneously, or wait until their run finishes.
	2.	If you really want to run a second notebook in parallel
	- It will share GPU 0 with both jobs.
	- But performance will tank since the GPU is already at 100%.
	- It may even crash if both jobs together exceed 24 GB VRAM.
	3.	Alternative
	- Run the second notebook in CPU mode (much slower, but safe).
	- Or, if your workstation has multiple GPUs (but here it looks like only 1 is installed), you’d assign another GPU like gpu_num=1.


So right now, since GPU 0 is maxed out, I’d recommend waiting until your colleague’s job finishes before launching a second training.

